# Seleção de participantes - Criação religiosa entre não religiosos (yes x no)

Análise exploratória do paper: entre os não religiosos, compara quem foi criado em ambiente religioso com quem não foi (análises `Non_religious_upbringing/Filtered`, Figura 4). (Para o pareamento por criação religiosa com toda a amostra, ver `SelectingParticipants_ReligiousUpbringing.ipynb`.)

**Regra de pareamento**

* Conjunto de partida: apenas não religiosos (`religious_now == 'no'`, 694 participantes).
* Âncoras: `religious_upbringing == 'yes'` (não religiosos criados em ambiente religioso), percorridas na ordem da planilha.
* Para cada âncora, candidatos com `religious_upbringing == 'no'` (não religiosos não criados em ambiente religioso) e mesmos `sex`, `handedness`, `education_level`.
* Escolhe-se o candidato com menor `|Δage| + |Δbackground_SES|` (coluna `dist`), sem reposição.
* Resultado: 141 + 141 = 282. 
* Produz `selected_participants-RU-RN-N.xlsx`.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats

# Planilha com os 928 participantes do ID1000 (participants.tsv do OpenNeuro ds003097 + colunas usadas no projeto)
PARTICIPANTS_CSV = "https://docs.google.com/spreadsheets/d/13n8f5tILuIVF6Cxi2izX020_fYNjNNZk/export?format=csv"

# Pasta local de dados do projeto (a que contém ID1000Files/). Rode o notebook a partir dela ou ajuste aqui.
DATA_ROOT = "."
OUTPUT_DIR = DATA_ROOT

OUTPUT_XLSX = os.path.join(OUTPUT_DIR, "selected_participants-RU-RN-N.xlsx")

df = pd.read_csv(PARTICIPANTS_CSV)
df.shape

(928, 31)

Identificar sujeitos que não têm funcional

In [2]:
# Remove sujeitos que têm anat/ mas não têm func/ (sem fMRI funcional): 928 -> 881
base_dir = os.path.join(DATA_ROOT, "ID1000Files", "raw")

anat_only = []
for dataset in os.listdir(base_dir):
    dataset_dir = os.path.join(base_dir, dataset)
    if not os.path.isdir(dataset_dir):
        continue
    subs = [d for d in os.listdir(dataset_dir) if d.startswith("sub-") and os.path.isdir(os.path.join(dataset_dir, d))]
    for subj in subs:
        anat_dir = os.path.join(dataset_dir, subj, "anat")
        func_dir = os.path.join(dataset_dir, subj, "func")
        has_anat = os.path.isdir(anat_dir) and any(os.path.isfile(os.path.join(anat_dir, f)) for f in os.listdir(anat_dir))
        has_func = os.path.isdir(func_dir) and any(os.path.isfile(os.path.join(func_dir, f)) for f in os.listdir(func_dir))
        if has_anat and not has_func:
            anat_only.append(subj)

df = df[~df["participant_id"].isin(anat_only)]
print("sujeitos sem func:", len(anat_only), "| pool:", len(df))

sujeitos sem func: 47 | pool: 881


## Pareamento

In [3]:
# Pareamento 1:1 guloso, sem reposição
pool = df[df["religious_now"] == "no"].copy()   # só não religiosos (694)
ancoras = pool[pool["religious_upbringing"] == "yes"].copy()      # não religiosos criados em ambiente religioso
candidatos_pool = pool[pool["religious_upbringing"] == "no"].copy()   # não religiosos não criados em ambiente religioso

EXATAS = ['sex', 'handedness', 'education_level']   # variáveis pareadas exatamente

pares = []
for idx, row in ancoras.iterrows():
    candidatos = candidatos_pool
    for col in EXATAS:
        candidatos = candidatos[candidatos[col] == row[col]]
    if candidatos.empty:
        continue
    candidatos = candidatos.copy()

    # Distância combinando idade e SES
    candidatos["dist"] = (
        (candidatos["age"] - row["age"]).abs() +
        (candidatos["background_SES"] - row["background_SES"]).abs()
    )
    mais_proximo = candidatos.sort_values("dist").iloc[0]

    pares.append(row)
    pares.append(mais_proximo)

    # Remove o já emparelhado
    candidatos_pool = candidatos_pool.drop(mais_proximo.name)

df_pares = pd.DataFrame(pares).reset_index(drop=True)
df_final = df_pares.sort_values(by=["age"])
print(df_final["religious_upbringing"].value_counts())
df_final

religious_upbringing
yes    141
no     141
Name: count, dtype: int64


,participant_id,age,sex,handedness,BMI,education_level,background_SES,IST_fluid,IST_memory,IST_crystallised,...,sexual_attraction_F,gender_identity_M,gender_identity_F,religious_upbringing,religious_now,religious_importance,DWI_TR_run1,DWI_TR_run2,DWI_TR_run3,dist
10,sub-0038,20.00,male,right,20,medium,3.0,110.0,51.0,32.0,...,7.0,6.0,2.0,yes,no,NaN,6.312,6.312,6.312,NaN
98,sub-0379,20.00,female,right,30,high,4.0,131.0,62.0,45.0,...,NaN,NaN,NaN,yes,no,NaN,6.374,6.374,6.374,NaN
40,sub-0139,20.00,male,right,20,medium,5.0,114.0,55.0,40.0,...,7.0,6.0,1.0,yes,no,NaN,6.311,6.311,6.311,NaN
11,sub-0803,20.25,male,right,20,medium,3.0,120.0,49.0,38.0,...,1.0,7.0,1.0,no,no,NaN,6.374,6.374,6.374,0.25
197,sub-0753,20.25,female,right,35,medium,2.0,74.0,41.0,31.0,...,NaN,NaN,NaN,no,no,NaN,6.312,6.312,6.312,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,sub-0147,25.75,male,right,26,medium,2.0,121.0,52.0,43.0,...,7.0,7.0,1.0,no,no,NaN,6.311,6.311,6.311,0.25
144,sub-0489,25.75,female,right,23,medium,5.0,76.0,41.0,30.0,...,NaN,NaN,NaN,yes,no,NaN,6.312,6.312,6.312,NaN
238,sub-0808,26.00,male,right,18,medium,2.0,72.0,51.0,31.0,...,2.0,5.0,1.0,yes,no,NaN,6.311,6.311,6.311,NaN
228,sub-0773,26.00,female,right,25,high,4.0,91.0,46.0,39.0,...,NaN,NaN,NaN,yes,no,NaN,6.311,6.311,6.311,NaN


## Verificação do balanço

In [4]:
# Balanço entre os grupos pareados
g1 = df_final[df_final["religious_upbringing"] == "yes"]
g2 = df_final[df_final["religious_upbringing"] == "no"]
for var in ["age", "background_SES", "IST_intelligence_total"]:
    t, p = stats.ttest_ind(g1[var].dropna(), g2[var].dropna(), equal_var=True)
    print(f"{var:24s} yes={g1[var].mean():.2f}  no={g2[var].mean():.2f}  t={t:.3f}  p={p:.3f}")
for var in EXATAS:
    print(var, "idêntico entre grupos:", (g1[var].value_counts().sort_index() == g2[var].value_counts().sort_index()).all())

age                      yes=22.93  no=22.90  t=0.150  p=0.881
background_SES           yes=4.09  no=4.06  t=0.180  p=0.858
IST_intelligence_total   yes=212.18  no=203.10  t=1.989  p=0.048
sex idêntico entre grupos: True
handedness idêntico entre grupos: True
education_level idêntico entre grupos: True


## Gravação

In [6]:
df_final.to_excel(OUTPUT_XLSX, index=False)